### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, Dense, GRU, Input, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras import Sequential

### Data Gathering

In [34]:
df = pd.read_csv(r"D:\Velocity\Datasets\NLP_Database\all_tickets_processed_improved_v3.csv")
df

,Document,Topic_group
0,connection with icon icon dear please setup ic...,Hardware
1,work experience user work experience user hi w...,Access
2,requesting for meeting requesting meeting hi p...,Hardware
3,reset passwords for external accounts re expir...,Access
4,mail verification warning hi has got attached ...,Miscellaneous
...,...,...
47832,git space for a project issues with adding use...,Access
47833,error sent july error hi guys can you help out...,Miscellaneous
47834,connection issues sent tuesday july connection...,Hardware
47835,error cube reports sent tuesday july error hel...,HR Support


In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47837 entries, 0 to 47836
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Document     47837 non-null  object
 1   Topic_group  47837 non-null  object
dtypes: object(2)
memory usage: 747.6+ KB


In [36]:
df['Topic_group'].value_counts()

Topic_group
Hardware                 13617
HR Support               10915
Access                    7125
Miscellaneous             7060
Storage                   2777
Purchase                  2464
Internal Project          2119
Administrative rights     1760
Name: count, dtype: int64

### Encoding

In [5]:
encoder = LabelEncoder()
df['Topic_group'] = encoder.fit_transform(df['Topic_group'])

### Data Preprocessing

In [6]:
max_len = 100
vocab_size = 5000

In [7]:
max_len = 100
vocab_size = 5000

tokenizer = Tokenizer(num_words=vocab_size, lower=True, oov_token='<OOV>')
tokenizer.fit_on_texts(df['Document'])

In [8]:
sequence = tokenizer.texts_to_sequences(df['Document'])

### Define X and Y

In [9]:
sequences = tokenizer.texts_to_sequences(df['Document'])
x = pad_sequences(sequences, maxlen=max_len, padding='post')

In [10]:
y = df['Topic_group']

### Train_Test_Split

In [11]:
x_train, x_test,y_train,y_test = train_test_split(x, y, test_size=0.2, random_state=42)

### Model Building

In [12]:
model = Sequential()

model.add(Input(shape=(max_len,)))
model.add(Embedding(input_dim=vocab_size, output_dim=64))

# Adding Bidirectional and Dropout
model.add(Bidirectional(GRU(units=64, return_sequences=False)))
model.add(Dropout(0.2))

# 8 units for 8 unique classes in Topic_group
model.add(Dense(units=8, activation='softmax'))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 100, 64)             │         320,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ (None, 128)                 │          49,920 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 8)                   │           1,032 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 370,952 (1.42 MB)

 Trainable params: 370,952 (1.42 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
early_stop = EarlyStopping(monitor= "val_loss", patience= 3, restore_best_weights=True)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs= 20, batch_size= 64, callbacks= [early_stop] )

Epoch 1/20
598/598 ━━━━━━━━━━━━━━━━━━━━ 80s 123ms/step - accuracy: 0.7336 - loss: 0.8099 - val_accuracy: 0.8264 - val_loss: 0.5148
Epoch 2/20
598/598 ━━━━━━━━━━━━━━━━━━━━ 73s 121ms/step - accuracy: 0.8631 - loss: 0.4154 - val_accuracy: 0.8498 - val_loss: 0.4463
Epoch 3/20
598/598 ━━━━━━━━━━━━━━━━━━━━ 81s 119ms/step - accuracy: 0.8873 - loss: 0.3374 - val_accuracy: 0.8473 - val_loss: 0.4456
Epoch 4/20
598/598 ━━━━━━━━━━━━━━━━━━━━ 78s 112ms/step - accuracy: 0.9013 - loss: 0.2919 - val_accuracy: 0.8511 - val_loss: 0.4573
Epoch 5/20
598/598 ━━━━━━━━━━━━━━━━━━━━ 75s 125ms/step - accuracy: 0.9132 - loss: 0.2555 - val_accuracy: 0.8453 - val_loss: 0.4799
Epoch 6/20
598/598 ━━━━━━━━━━━━━━━━━━━━ 78s 118ms/step - accuracy: 0.9237 - loss: 0.2251 - val_accuracy: 0.8513 - val_loss: 0.4981


### Evaluation

In [15]:
loss, accuracy = model.evaluate(x_test,y_test)
accuracy

299/299 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.8473 - loss: 0.4456


0.8473035097122192

In [16]:
loss, accuracy = model.evaluate(x_train ,y_train)
accuracy

1196/1196 ━━━━━━━━━━━━━━━━━━━━ 24s 20ms/step - accuracy: 0.9187 - loss: 0.2555


0.9186809062957764

### Testing on Single Text

In [25]:
text1 = "access to dear colleagues help by providing tool best regards find out more about experience lead blvd floor"

In [23]:
# 1. Preprocess text
seq = tokenizer.texts_to_sequences([text1])
pad_seq = pad_sequences(seq, maxlen=max_len, padding="post")

# 2. Predict probabilities across all classes
predictions = model.predict(pad_seq)

# 3. Find highest probability index
predicted_class_index = np.argmax(predictions)

# 4. Map index to Topic Group name
predicted_topic = encoder.inverse_transform([predicted_class_index])[0]

# 5. Output results
print(f"Probabilities: {predictions}")
print(f"Predicted Class Index: {predicted_class_index}")
print(f"Predicted Topic Group: {predicted_topic}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Probabilities: [[1.5233704e-02 7.2045752e-04 8.0456942e-01 1.7444666e-01 2.8483849e-04
  4.5894441e-04 3.8792708e-04 3.8980192e-03]]
Predicted Class Index: 2
Predicted Topic Group: HR Support


In [24]:
text2 = "write access to commercial client management code project setup managed extra services re updated commercial client code setup hi although code due active till st july enter draft budget appears writable code please give write update these figures thanks technical programme id canary updated commercial client code setup resolved fellow think query logged has resolved did well details reference summary commercial client code setup description type setup setup commercial commercial client code commercial client code more resolution code good provided resolution response issue persist recur resolved please best provide assistance matter how how how warm regards ext ref msg"

# 1. Preprocess text
seq = tokenizer.texts_to_sequences([text2])
pad_seq = pad_sequences(seq, maxlen=max_len, padding="post")

# 2. Predict probabilities across all classes
predictions = model.predict(pad_seq)

# 3. Find highest probability index
predicted_class_index = np.argmax(predictions)

# 4. Map index to Topic Group name
predicted_topic = encoder.inverse_transform([predicted_class_index])[0]

# 5. Output results
print(f"Probabilities: {predictions}")
print(f"Predicted Class Index: {predicted_class_index}")
print(f"Predicted Topic Group: {predicted_topic}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
Probabilities: [[3.9162944e-05 8.1738566e-05 1.2091968e-02 2.0563804e-05 9.7671276e-01
  5.7750492e-04 8.6002494e-04 9.6162613e-03]]
Predicted Class Index: 4
Predicted Topic Group: Internal Project


### Save Model

In [48]:
# Returns an array of all 8 class names
all_classes = label_encoder.classes_
print(all_classes)

['Access' 'Administrative rights' 'HR Support' 'Hardware'
 'Internal Project' 'Miscellaneous' 'Purchase' 'Storage']


In [49]:
# Extract class names from the encoder
class_names = label_encoder.classes_.tolist()

# Save the model
model.save("Ticket_clf_model.keras")

# Save class names to JSON
with open("topic_classes.json", "w") as f:
    json.dump(class_names, f)